<a href="https://colab.research.google.com/github/osh9149/2025_DataScience/blob/main/%EA%B3%A0%EC%96%91%EC%9D%B4_vs_%EA%B0%95%EC%95%84%EC%A7%80_%EC%9D%B4%EB%AF%B8%EC%A7%80_%EB%B6%84%EB%A5%98_(CNN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐱🐶 고양이 vs 강아지 이미지 분류 (CNN)

이 노트북은 CNN(합성곱 신경망)을 사용해 고양이와 강아지 이미지를 분류하는 모델을 학습하는 실습입니다.

**사용 데이터**: TensorFlow의 `cats_vs_dogs` 데이터셋
**사용 기술**: Conv2D, MaxPooling, Flatten, Dense 등

※ 이 데이터는 다운로드 시간이 조금 걸릴 수 있습니다.

In [ ]:
# TensorFlow 및 관련 라이브러리 불러오기
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory

## 1. 데이터 다운로드 및 준비

In [ ]:
# 고양이/강아지 데이터 다운로드
url = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'
path_to_zip = tf.keras.utils.get_file('cats_and_dogs_filtered.zip', origin=url, extract=True)
base_dir = os.path.join(os.path.dirname(path_to_zip), 'cats_and_dogs_filtered')

In [ ]:
# 학습 및 검증 데이터셋 불러오기
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')

batch_size = 32
img_size = (160, 160)

train_ds = image_dataset_from_directory(train_dir,
                                        shuffle=True,
                                        batch_size=batch_size,
                                        image_size=img_size)

val_ds = image_dataset_from_directory(validation_dir,
                                      shuffle=True,
                                      batch_size=batch_size,
                                      image_size=img_size)

## 2. 데이터 성능 최적화

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 3. CNN 모델 구성 및 학습

In [ ]:
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(160, 160, 3)),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

## 4. 모델 학습

In [ ]:
epochs = 5
history = model.fit(train_ds,
                    validation_data=val_ds,
                    epochs=epochs)

## 5. 학습 결과 시각화

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()